# Self-consistency

**Definition** (sampling-based self-consistency, Wang et al., 2022): the same question with the same documents is answered
**5 times at temperature 0.7** (each sample with its own seed). If the 5 answers agree, the model is consistent;
if they spread over the true and the false answer, the conflict in the documents shows up as uncertainty.

- Setting: controlled poisoning, baseline prompt, no source labels, m = 0..4.
- Subset: 100 questions known by at least one model + 100 known by neither (from the final selection), for cost reasons.
- `agreement` = share of the 5 samples equal to the most frequent answer (1.0 = all identical).

In [1]:
import json
import random
from pathlib import Path

import pandas as pd

from ragtrust.analysis import consistency_by_question, known_ids
from ragtrust.data import pick_fewshot_examples
from ragtrust.experiment import cached_run, run_self_consistency
from ragtrust.llm import OllamaLLM
from ragtrust.poison import load_poison_set, select_items
from ragtrust.runs import SELECTION_PATH, load_selection

MODELS = ["llama3.1:8b", "qwen2.5:7b"]
TAG = "final"   # part of every result file name
SEED = 0
N_PER_GROUP = 100
RESULTS = Path("../results")
POISON_PATH = Path(f"../data/poison/gemma4-12b_{TAG}.jsonl")
EXCLUDED_PATH = Path("../data/poison/excluded.json")

examples = load_selection()
excluded = json.load(open(EXCLUDED_PATH)) if EXCLUDED_PATH.exists() else {}
items = select_items(load_poison_set(POISON_PATH), examples, excluded)

group = json.load(open(SELECTION_PATH))["group"]
rng = random.Random(SEED)
subset = []
for g in ["known", "unknown"]:
    candidates = sorted((it for it in items if group[it.id] == g), key=lambda it: it.id)
    subset += rng.sample(candidates, min(N_PER_GROUP, len(candidates)))
print(f"Subset: {len(subset)} questions")

Subset: 200 questions


In [ ]:
fewshot = pick_fewshot_examples(seed=SEED)
frames = []
for model in MODELS:
    llm = OllamaLLM(model=model, seed=SEED)
    path = RESULTS / f"self_consistency_{model.replace(':', '-')}_{TAG}.csv"
    run = lambda: run_self_consistency(llm, examples, subset, n_samples=5, temperature=0.7, fewshot=fewshot, seed=SEED)
    frames.append(cached_run(path, llm, run).assign(model=model))
samples = pd.concat(frames)
cons = consistency_by_question(samples)

## Results

In [3]:
# Mean agreement by model and poisoning level
cons.groupby(["model", "m"])["agreement"].mean().unstack("m").round(3)

m,0,1,2,3,4
model,,,,,
llama3.1:8b,0.900,0.854,0.782,0.784,0.838
qwen2.5:7b,0.954,0.933,0.901,0.928,0.940


In [4]:
# Known vs. unknown, using each model's own closed-book answers (notebook 00)
closed_book = pd.concat([pd.read_csv(RESULTS / f"selection_closed_book_{m.replace(':', '-')}_all.csv").assign(model=m)
                         for m in MODELS])
known = {(m, i) for m in MODELS for i in known_ids(closed_book, m)}
cons["known"] = [(m, i) in known for m, i in zip(cons["model"], cons["id"])]
cons.groupby(["model", "known", "m"])["agreement"].mean().unstack("m").round(3)

m                      0      1      2      3      4
model       known                                   
llama3.1:8b False  0.876  0.817  0.743  0.749  0.822
            True   0.941  0.916  0.849  0.843  0.865
qwen2.5:7b  False  0.953  0.932  0.897  0.922  0.944
            True   0.956  0.934  0.909  0.941  0.931

In [5]:
# Majority answer category by level
pd.crosstab([cons["model"], cons["m"]], cons["majority_category"], normalize="index").round(3)

majority_category  correct  hedged  other  poisoned
model       m                                      
llama3.1:8b 0        0.785   0.000  0.210     0.005
            1        0.725   0.010  0.210     0.055
            2        0.620   0.030  0.190     0.160
            3        0.400   0.010  0.190     0.400
            4        0.125   0.010  0.225     0.640
qwen2.5:7b  0        0.815   0.000  0.180     0.005
            1        0.795   0.000  0.170     0.035
            2        0.585   0.005  0.155     0.255
            3        0.295   0.005  0.165     0.535
            4        0.085   0.005  0.180     0.730